In [ ]:
"""OPTION A, CELL 1: BUILD THE TWO TRAINING FILES (no GPU work, about 1 minute).
Fold A = round 1 data + Swedish group A words. Its model is tested only on group B words.
Fold B = round 1 data + Swedish group B words. Its model is tested only on group A words.
So every Swedish word gets scored by a model that never saw it."""
import sys, json, hashlib
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/semeval2027/semeval2027_colab")
for folder in (PROJECT / "src", PROJECT / "notebooks"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
from project_utils import save_json
from semeval_task3.data import load_dataset

def section(t):
    print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78, flush=True)

R1_DATA = PROJECT / "outputs/runs/step07_pilot_dataset/20261105T223146Z_1dd5a63b"
ENS_DECISION = PROJECT / "outputs/runs/step15_seed_ensemble/ensemble_decision.json"
DATA = PROJECT / "outputs/runs/step16_option_a/data"
SYSTEM = "You are an expert lexicographer of historical Swedish."

def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def read_rows(path):
    with Path(path).open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

def swedish_messages(u):
    """Exactly the same prompt our Swedish evaluation has always used."""
    return [dict(role="system", content=SYSTEM),
            dict(role="user", content=(f"Target word (lemma): {u.word}\n"
                                       f"Sense definition (Swedish): {u.definition}\n"
                                       f"Sentence from {u.year}: {u.marked_text()}\n\n"
                                       "Does the word marked with <t> and </t> in the sentence express the sense given by the definition? "
                                       "Answer only Yes or No."))]

# ---------------------------------------------------------------------------
section("1. ROUND 1 TRAINING DATA (Russian + English), unchanged")
m1 = json.loads((R1_DATA / "manifest.json").read_text())
train_path = R1_DATA / "pilot_train.jsonl"
assert digest(train_path) == m1["sha256"]["pilot_train.jsonl"], "Round 1 training file changed: STOP and paste this output."
round1 = read_rows(train_path)
assert len(round1) == 8510, f"Expected 8,510 round 1 examples, found {len(round1)}: STOP."
assert not any(r.get("language") == "SV" for r in round1), "Round 1 already contains Swedish rows: STOP."
r1_yes = sum(int(r["label"]) for r in round1)
print(f"Round 1 file : hash PASS | {len(round1):,} examples | Yes share {r1_yes / len(round1):.1%}")

# ---------------------------------------------------------------------------
section("2. SPLIT THE 10 SWEDISH DEV WORDS INTO TWO GROUPS (fixed rule: alphabetical, every other word)")
sv = load_dataset(PROJECT / "work/data/dev", with_gold=True)
gold = sv.require_gold()
assert len(sv.usages) == 1983 and set(gold) == {u.key for u in sv.usages}, "Swedish dev data is not the expected 1,983 usages: STOP."
assert all(int(v) in (0, 1) for v in gold.values())
words = sorted({u.word for u in sv.usages})
assert len(words) == 10, f"Expected 10 Swedish words, found {len(words)}: STOP."
GROUPS = {"A": words[0::2], "B": words[1::2]}
for g in ("A", "B"):
    us = [u for u in sv.usages if u.word in GROUPS[g]]
    yes = sum(int(gold[u.key]) for u in us)
    print(f"Group {g}: {', '.join(GROUPS[g])}")
    print(f"         {len(us):,} usages | Yes share {yes / len(us):.1%}")

# ---------------------------------------------------------------------------
section("3. BUILD THE TWO TRAINING FILES")
round1_ids = {str(r["id"]) for r in round1}
texts, counts = {}, {}
for g, other in (("A", "B"), ("B", "A")):
    sw = [dict(id="|".join(u.key), language="SV", word=u.word, label=int(gold[u.key]),
               source="SemEval-2027 Task 3 Swedish dev data (organizers)", messages=swedish_messages(u))
          for u in sv.usages if u.word in GROUPS[g]]
    assert not round1_ids & {r["id"] for r in sw}, "ID clash between round 1 and Swedish rows: STOP."
    assert {r["word"] for r in sw} == set(GROUPS[g]) and not {r["word"] for r in sw} & set(GROUPS[other]), "Leak check failed: STOP."
    rows = round1 + sw
    texts[f"train_{g}.jsonl"] = "".join(json.dumps(r, ensure_ascii=False, sort_keys=True) + "\n" for r in rows)
    counts[g] = dict(total=len(rows), round1=len(round1), swedish=len(sw), swedish_yes=sum(r["label"] for r in sw),
                     updates=-(-len(rows) // 16), trained_on=GROUPS[g], tested_on=GROUPS[other])
    print(f"Fold {g}: {len(rows):,} examples = {len(round1):,} round 1 + {len(sw):,} Swedish ({len(sw) / len(rows):.1%} Swedish) "
          f"| {counts[g]['updates']} updates | tested later on group {other} only")

# ---------------------------------------------------------------------------
section("4. THE RULE, WRITTEN DOWN BEFORE ANY TRAINING")
ens = json.loads(ENS_DECISION.read_text())
ENS_SV = ens["swedish"]["ensemble (average of 3)"]
R1_SV = ens["swedish"]["seed 2027 (round 1)"]
RULE = dict(
    health="each fold model: loss went down, adapter differs from round 1, all scores finite, both Yes and No predicted",
    rule_1a=f"option A Swedish macro-F1 (nested, every word scored by a model that never saw it) >= {ENS_SV + 0.02:.4f} (system 2 {ENS_SV:.4f} + 0.02)",
    rule_1b="better than system 2 on at least 6 of the 10 Swedish words",
    rule_2="each fold model loses at most 0.02 AUC and at most 0.02 macro-F1 vs round 1 on EN, RU, IT, ES, NL",
    if_all_pass="train one final model on round 1 data + all 10 Swedish words for January",
    if_any_fail="drop option A; system 1 and system 2 stay exactly as they are")
for k, v in RULE.items():
    print(f"{k:<12}: {v}")

# ---------------------------------------------------------------------------
section("5. SAVE")
DATA.mkdir(parents=True, exist_ok=True)
sha = {name: hashlib.sha256(text.encode("utf-8")).hexdigest() for name, text in texts.items()}
manifest_path = DATA / "manifest.json"
if manifest_path.exists():
    old = json.loads(manifest_path.read_text())
    assert old["sha256"] == sha and old["groups"] == GROUPS, "Option A files already exist and differ from this build: STOP and paste this output."
    for name in texts:
        assert digest(DATA / name) == sha[name], f"{name} on Drive was changed: STOP."
    print("Files were already built earlier and are identical: nothing rewritten.")
else:
    for name, text in texts.items():
        (DATA / name).write_text(text, encoding="utf-8")
        assert digest(DATA / name) == sha[name], f"{name}: written file does not match: STOP."
    save_json(manifest_path, dict(created_utc=datetime.now(timezone.utc).isoformat(), groups=GROUPS, sha256=sha, counts=counts,
                                  round1_train_sha256=m1["sha256"]["pilot_train.jsonl"], system1_swedish=R1_SV,
                                  system2_swedish=ENS_SV, rule=RULE, seed=2027))
    print("Written: train_A.jsonl, train_B.jsonl, manifest.json")
print(f"Folder: {DATA}")

section("SUMMARY")
print(f"Fold A trains on {', '.join(GROUPS['A'])}  -> tested on {', '.join(GROUPS['B'])}")
print(f"Fold B trains on {', '.join(GROUPS['B'])}  -> tested on {', '.join(GROUPS['A'])}")
print(f"To pass, option A needs Swedish >= {ENS_SV + 0.02:.4f} AND no other language worse.")
print("Nothing was trained and no existing file was changed. Paste this whole output.")

In [ ]:
"""OPTION A, CELL 2: TRAIN THE TWO FOLD MODELS, THEN SCORE THEM (G4 runtime, about 2 hours 15 minutes in total).
Same recipe and seed as round 1. The only difference: each fold also trains on 5 Swedish dev words.
If Colab disconnects: reconnect to a G4 and run THIS cell again. It resumes from the last save (every 100 updates)."""
import sys, gc, json, math, time, uuid, random, shutil, hashlib, subprocess, importlib
from importlib.metadata import version
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/semeval2027/semeval2027_colab")
for folder in (PROJECT / "src", PROJECT / "notebooks"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft"], check=True)
importlib.invalidate_caches()
import numpy as np, torch, torch.nn.functional as F
from peft import LoraConfig, PeftModel, get_peft_model
from peft.import_utils import is_torchao_available
from project_utils import save_json
from semeval_task3.data import load_dataset

def section(t):
    print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78, flush=True)

DEVICE = "cuda"
MODEL_ID = "google/gemma-4-31B-it"
REVISION = "842da3794eaa0b77d5f08bae87a17459d91ff475"
SEED = 2027                                   # same seed as round 1
FOLDS = {"A": "B", "B": "A"}                  # trained on group -> tested on group
ROOT = PROJECT / "outputs/runs/step16_option_a"
DATA = ROOT / "data"
ENRU_EVAL = PROJECT / "outputs/runs/step07_pilot_evaluation/20261105T223700Z_5259f45a"
XL_EVAL = PROJECT / "outputs/runs/step11_xlwsd_evaluation_sets/20261106T032028Z_aa4e2120"
EFFECTIVE_BATCH, LR, SAVE_EVERY, N_TARGET_LAYERS = 16, 1e-4, 100, 230
YES_ID, NO_ID = 10784, 3771
SYSTEM = "You are an expert lexicographer of historical Swedish."

def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def read_rows(path):
    with Path(path).open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

def swedish_messages(u):
    return [dict(role="system", content=SYSTEM),
            dict(role="user", content=(f"Target word (lemma): {u.word}\n"
                                       f"Sense definition (Swedish): {u.definition}\n"
                                       f"Sentence from {u.year}: {u.marked_text()}\n\n"
                                       "Does the word marked with <t> and </t> in the sentence express the sense given by the definition? "
                                       "Answer only Yes or No."))]

# ---------------------------------------------------------------------------
section("1. CHECK EVERY INPUT FILE BEFORE TOUCHING THE MODEL")
is_torchao_available.cache_clear()
assert is_torchao_available() is False, "torchao is still visible to peft: Runtime > Restart session, then run this cell again."
manifest = json.loads((DATA / "manifest.json").read_text())
GROUPS = manifest["groups"]
training = {}
for g in FOLDS:
    p = DATA / f"train_{g}.jsonl"
    assert digest(p) == manifest["sha256"][p.name], f"{p.name} changed since cell 1: STOP and paste this output."
    rows = read_rows(p)
    assert len(rows) == manifest["counts"][g]["total"]
    for r in rows:
        assert int(r["label"]) in (0, 1) and r["messages"][-1]["role"] == "user"
        assert all(m["role"] in ("system", "user") for m in r["messages"])
    training[g] = rows
    print(f"Fold {g} training file : hash PASS | {len(rows):,} examples")

enru_manifest = json.loads((ENRU_EVAL / "manifest.json").read_text())
xl_manifest = json.loads((XL_EVAL / "manifest.json").read_text())
eval_sets = {}
for name in ("EN_wic_dev", "RU_axolotl_test_derived"):
    assert digest(ENRU_EVAL / f"{name}.jsonl") == enru_manifest["output_sha256"][f"{name}.jsonl"], f"{name} changed"
    eval_sets[name] = read_rows(ENRU_EVAL / f"{name}.jsonl")
for lang in ("IT", "ES", "NL"):
    name = f"{lang}_xlwsd_test"
    assert digest(XL_EVAL / f"{name}.jsonl") == xl_manifest["files"][f"{name}.jsonl"], f"{name} changed"
    eval_sets[name] = read_rows(XL_EVAL / f"{name}.jsonl")
print(f"Other-language test sets: hash PASS | { {k: len(v) for k, v in eval_sets.items()} }")

sv = load_dataset(PROJECT / "work/data/dev", with_gold=False)
assert len(sv.usages) == 1983
by_id = {"|".join(u.key): u for u in sv.usages}
sv_test = {}
for g, other in FOLDS.items():
    trained = {r["id"]: r for r in training[g] if r.get("language") == "SV"}
    assert {r["word"] for r in trained.values()} == set(GROUPS[g]), f"Fold {g}: Swedish training words are not group {g}: STOP."
    for i, r in trained.items():
        assert r["messages"] == swedish_messages(by_id[i]), "Swedish training prompt differs from the evaluation prompt: STOP."
    sv_test[g] = [dict(id=i, messages=swedish_messages(u)) for i, u in by_id.items() if u.word in GROUPS[other]]
    assert not {by_id[r["id"]].word for r in sv_test[g]} & set(GROUPS[g]), f"Fold {g}: a test word is also a training word: STOP."
    print(f"Fold {g}: trains on {len(trained):,} Swedish usages ({', '.join(GROUPS[g])}) | "
          f"tested on {len(sv_test[g]):,} unseen usages ({', '.join(GROUPS[other])}) | leak check PASS")
assert len(sv_test["A"]) + len(sv_test["B"]) == 1983
print("Swedish prompts in training are identical to the evaluation prompts: PASS")

# ---------------------------------------------------------------------------
section("2. GEMMA 4 31B (bf16)")
if "gemma_model" in globals() or "full_model" in globals() or "BASE" in globals():
    print("Gemma is already loaded in this session: reusing it.")
else:
    from transformers import AutoProcessor, AutoModelForMultimodalLM
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} | {props.total_memory / 2**30:.0f} GiB | transformers {version('transformers')}")
    assert props.total_memory / 2**30 > 70, "Gemma 31B training needs a GPU with more than 70 GB: use the G4."
    print("Loading Gemma (about 5 minutes on the G4) ...", flush=True)
    gemma_processor = AutoProcessor.from_pretrained(MODEL_ID, revision=REVISION)
    gemma_tokenizer = gemma_processor.tokenizer
    gemma_model = AutoModelForMultimodalLM.from_pretrained(
        MODEL_ID, revision=REVISION, dtype=torch.bfloat16, device_map={"": 0}, attn_implementation="sdpa")
    print(f"Loaded. GPU memory in use: {torch.cuda.memory_allocated() / 2**30:.1f} GiB")
gemma_tokenizer.padding_side = "left"
assert gemma_tokenizer.encode("Yes", add_special_tokens=False) == [YES_ID]
assert gemma_tokenizer.encode("No", add_special_tokens=False) == [NO_ID]

def plain_base():
    """The original Gemma with every adapter removed (unload() removes adapters without merging them)."""
    current = None
    for name in ("full_model", "BASE", "gemma_model"):
        if globals().get(name) is not None:
            current = globals()[name]
            break
    base = current.unload() if isinstance(current, PeftModel) else current
    for name in ("full_model", "gemma_model", "model", "full_optimizer", "params"):
        globals().pop(name, None)
    assert base.config._name_or_path == MODEL_ID and not any("lora_" in n for n, _ in base.named_parameters())
    gc.collect()
    torch.cuda.empty_cache()
    return base

def encode(batch):
    inputs = gemma_processor.apply_chat_template(
        [r["messages"] for r in batch], tokenize=True, return_dict=True, return_tensors="pt",
        add_generation_prompt=True, enable_thinking=False,
        processor_kwargs={"padding": True, "truncation": False}).to(DEVICE)
    assert inputs["attention_mask"][:, -1].all(), "Padding is not on the left."
    return inputs

def score_rows(model, rows, destination, tag):
    """Resumable scoring: labels never enter the model; saved to Drive every 100 batches."""
    identity = hashlib.sha256(json.dumps([(str(r["id"]), r["messages"]) for r in rows],
                                         ensure_ascii=False, sort_keys=True).encode()).hexdigest()
    if destination.exists():
        saved = json.loads(destination.read_text())
        assert saved["input_sha256"] == identity and saved["model_tag"] == tag, f"{destination.name}: saved scores belong to other inputs."
    else:
        saved = dict(input_sha256=identity, model_tag=tag, scores={})
    pending = [r for r in rows if str(r["id"]) not in saved["scores"]]
    model.eval()
    t0 = time.perf_counter()
    try:
        with torch.inference_mode(), torch.autocast(DEVICE, dtype=torch.bfloat16):
            for b, i in enumerate(range(0, len(pending), 2), 1):
                batch = pending[i:i + 2]
                logits = model(**encode(batch), use_cache=False, logits_to_keep=1).logits[:, -1, :].float()
                assert torch.isfinite(logits).all()
                for r, margin in zip(batch, (logits[:, YES_ID] - logits[:, NO_ID]).tolist()):
                    saved["scores"][str(r["id"])] = dict(margin=float(margin))
                del logits
                if b % 100 == 0:
                    save_json(destination, saved)
    finally:
        save_json(destination, saved)
    assert set(saved["scores"]) == {str(r["id"]) for r in rows}
    if pending:
        print(f"      {tag}: {len(rows):,} scores ({(time.perf_counter() - t0) / 60:.1f} min)", flush=True)
    return saved

# ---------------------------------------------------------------------------
def train_fold(g):
    global BASE, full_model
    run = ROOT / f"fold_{g}"
    run.mkdir(parents=True, exist_ok=True)
    rows = training[g]
    total = math.ceil(len(rows) / EFFECTIVE_BATCH)
    warmup = math.ceil(0.05 * total)
    config = dict(model=MODEL_ID, revision=REVISION, seed=SEED, fold=g,
                  trained_on_swedish=GROUPS[g], tested_on_swedish=GROUPS[FOLDS[g]],
                  recipe="round 1 (step09) unchanged; training data = round 1 + 5 Swedish dev words",
                  examples=len(rows), train_sha256=manifest["sha256"][f"train_{g}.jsonl"], micro_batch=1,
                  effective_batch=EFFECTIVE_BATCH, learning_rate=LR, warmup_steps=warmup, steps=total,
                  schedule="linear warmup then linear decay", r=16, alpha=32, dropout=0.05,
                  target="language-model q/k/v/o", base_precision="BF16, no quantization",
                  checkpoint="final update (fixed before training)")
    if (run / "config.json").exists():
        assert json.loads((run / "config.json").read_text()) == json.loads(json.dumps(config)), \
            f"Fold {g}: settings changed since this run started. STOP and paste this message."
    else:
        save_json(run / "config.json", config)
    if (run / "COMPLETE.json").exists():
        print(f"Fold {g}: already complete, skipping.")
        return
    BASE = plain_base()
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    random.seed(SEED)
    np.random.seed(SEED)
    order = list(rows)
    random.Random(SEED).shuffle(order)
    targets = [n for n, m in BASE.named_modules() if isinstance(m, torch.nn.Linear) and "language_model" in n.split(".")
               and n.split(".")[-1] in ("q_proj", "k_proj", "v_proj", "o_proj")]
    assert len(targets) == N_TARGET_LAYERS, f"Unexpected number of attention layers: {len(targets)}"
    latest_file = run / "latest.json"
    final_dir = run / "adapter_final"
    state = None
    if (final_dir / "adapter_config.json").exists():
        full_model = PeftModel.from_pretrained(BASE, str(final_dir), is_trainable=False)
        step_done, log = total, json.loads((run / "training_log.json").read_text())
        print(f"Fold {g}: training already finished; continuing with scoring.")
    elif latest_file.exists():
        latest = Path(json.loads(latest_file.read_text())["directory"])
        full_model = PeftModel.from_pretrained(BASE, str(latest / "adapter"), is_trainable=True)
        state = torch.load(latest / "training_state.pt", map_location="cpu", weights_only=False)     # our own file
        step_done, log = state["step"], state["log"]
        print(f"Fold {g}: resuming after update {step_done} of {total}.")
    else:
        full_model = get_peft_model(BASE, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules=targets, bias="none"))
        step_done, log = 0, []
        assert all(torch.count_nonzero(p).item() == 0 for n, p in full_model.named_parameters() if "lora_B" in n)
        print(f"Fold {g}: fresh adapter attached (starts from zero).")

    if step_done < total:
        full_model.config.use_cache = False
        full_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        params = [p for p in full_model.parameters() if p.requires_grad]
        assert params and all("lora_" in n for n, p in full_model.named_parameters() if p.requires_grad)
        optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)
        if state:
            optimizer.load_state_dict(state["optimizer"])
            random.setstate(state["python_rng"])
            np.random.set_state(state["numpy_rng"])
            torch.set_rng_state(state["torch_rng"])
            if state.get("cuda_rng") is not None:
                torch.cuda.set_rng_state_all(state["cuda_rng"])
        del state

        def save_resume(step):
            folder = run / f"resume_{step:04d}_{uuid.uuid4().hex[:8]}"
            folder.mkdir()
            full_model.save_pretrained(folder / "adapter", safe_serialization=True)
            torch.save(dict(step=step, log=log, optimizer=optimizer.state_dict(), python_rng=random.getstate(),
                            numpy_rng=np.random.get_state(), torch_rng=torch.get_rng_state(),
                            cuda_rng=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None),
                       folder / "training_state.pt")
            previous = json.loads(latest_file.read_text())["directory"] if latest_file.exists() else None
            save_json(latest_file, {"directory": str(folder), "step": step})
            if previous and Path(previous).parent == run and Path(previous).name.startswith("resume_"):
                shutil.rmtree(previous, ignore_errors=True)
            save_json(run / "training_log.json", log)

        full_model.train()
        t0 = time.perf_counter()
        for step in range(step_done + 1, total + 1):
            batch_rows = order[(step - 1) * EFFECTIVE_BATCH:step * EFFECTIVE_BATCH]
            scale = step / warmup if step <= warmup else (total - step + 1) / (total - warmup)
            for pg in optimizer.param_groups:
                pg["lr"] = LR * scale
            optimizer.zero_grad(set_to_none=True)
            loss_sum, correct = 0.0, 0
            for r in batch_rows:
                answer = torch.tensor([YES_ID if int(r["label"]) else NO_ID], device=DEVICE)
                with torch.autocast(DEVICE, dtype=torch.bfloat16):
                    logits = full_model(**encode([r]), use_cache=False, logits_to_keep=1).logits[:, -1, :]
                loss = F.cross_entropy(logits.float(), answer)
                assert torch.isfinite(loss), "Non-finite loss: STOP and paste this message. The last saved checkpoint is intact."
                (loss / len(batch_rows)).backward()
                loss_sum += float(loss.detach())
                correct += int((logits[0, YES_ID] > logits[0, NO_ID]).item() == bool(int(r["label"])))
                del logits, loss, answer
            norm = float(torch.nn.utils.clip_grad_norm_(params, 1.0, error_if_nonfinite=True))
            optimizer.step()
            log.append(dict(step=step, loss=loss_sum / len(batch_rows), batch_accuracy=correct / len(batch_rows),
                            grad_norm=norm, lr=LR * scale))
            if step == 1 or step % 50 == 0 or step == total:
                minutes = (time.perf_counter() - t0) / 60
                print(f"   fold {g} | update {step:>3}/{total}: loss {loss_sum / len(batch_rows):.4f} | "
                      f"accuracy {correct / len(batch_rows):.2f} | about {minutes / (step - step_done) * (total - step):.0f} min of training left",
                      flush=True)
            if step % SAVE_EVERY == 0 and step < total:
                save_resume(step)
        full_model.save_pretrained(final_dir, safe_serialization=True)
        save_json(run / "training_log.json", log)
        optimizer.zero_grad(set_to_none=True)
        del optimizer, params
        if latest_file.exists():
            previous = Path(json.loads(latest_file.read_text())["directory"])
            if previous.parent == run and previous.name.startswith("resume_"):
                shutil.rmtree(previous, ignore_errors=True)
            latest_file.unlink()
        gc.collect()
        torch.cuda.empty_cache()
        losses = [x["loss"] for x in log]
        print(f"Fold {g}: training done | mean loss first 50 updates {np.mean(losses[:50]):.3f} -> last 50 updates {np.mean(losses[-50:]):.3f}")

    print(f"Fold {g}: scoring on 5 other-language test sets + {len(sv_test[g]):,} unseen Swedish usages (about 9 minutes)", flush=True)
    (run / "eval_scores").mkdir(exist_ok=True)
    for name, rows_to_score in list(eval_sets.items()) + [("SV_heldout", sv_test[g])]:
        score_rows(full_model, rows_to_score, run / "eval_scores" / f"{name}.json", f"fold_{g}_{name}")
    save_json(run / "COMPLETE.json", dict(fold=g, adapter=str(final_dir), adapter_sha256=digest(final_dir / "adapter_model.safetensors"),
                                          trained_on_swedish=GROUPS[g], tested_on_swedish=GROUPS[FOLDS[g]],
                                          eval_scores={name: f"eval_scores/{name}.json" for name in list(eval_sets) + ["SV_heldout"]}))
    print(f"Fold {g}: COMPLETE")
    BASE = plain_base()

for i, g in enumerate(("A", "B"), 3):
    section(f"{i}. FOLD {g}: TRAIN ON ROUND 1 + {', '.join(GROUPS[g]).upper()}, THEN SCORE (about 1 hour)")
    train_fold(g)

section("SUMMARY")
for g in FOLDS:
    print(f"Fold {g}: {'COMPLETE' if (ROOT / f'fold_{g}' / 'COMPLETE.json').exists() else 'NOT complete'}")
print(f"Everything is saved in: {ROOT}")
print("System 1 and system 2 are untouched. Nothing is decided yet.")
print("Next: paste this output, then run the decision cell (no GPU needed).")

In [ ]:
"""OPTION A, CELL 3: THE DECISION (no GPU work, about 2 minutes).
Combines the two fold models' scores on the Swedish words each one never saw, then applies the rule written in cell 1."""
import sys, json, hashlib, subprocess, importlib.util
from collections import defaultdict
from pathlib import Path
import numpy as np
from google.colab import drive

drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/semeval2027/semeval2027_colab")
for folder in (PROJECT / "src", PROJECT / "notebooks"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bcubed"], check=True)
from sklearn.metrics import roc_auc_score
from project_utils import save_json, frozen
from semeval_task3.data import load_dataset
import semeval_task3.cv as cv
from semeval_task3.cv import Candidate, nested_word_cv
from semeval_task3.evaluation import score as positive_score, word_bootstrap

def section(t):
    print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78, flush=True)

ROOT = PROJECT / "outputs/runs/step16_option_a"
ENS_ROOT = PROJECT / "outputs/runs/step15_seed_ensemble"
R1_RUN = PROJECT / "outputs/runs/step09_lora_full_pilot/20261105T232211Z_6b04a51b"
FROZEN = PROJECT / "outputs/frozen/primary_gemma4_31b_lora_macro.json"
ENRU_EVAL = PROJECT / "outputs/runs/step07_pilot_evaluation/20261105T223700Z_5259f45a"
XL_EVAL = PROJECT / "outputs/runs/step11_xlwsd_evaluation_sets/20261106T032028Z_aa4e2120"
XL_SCORES = PROJECT / "outputs/runs/step11_xlwsd_scores/adapter_6f3d236ac50e"
SCORER_DIR, SCORER_COMMIT = Path("/content/official-scorer-latest"), "0e69238"
GRID = tuple([-1000.0] + np.round(np.arange(-100, 100.001, 0.25), 2).tolist() + [1000.0])
EXTERNAL = ("EN_wic_dev", "RU_axolotl_test_derived", "IT_xlwsd_test", "ES_xlwsd_test", "NL_xlwsd_test")
FOLDS = {"A": "B", "B": "A"}
R1N, ENSN, OPTN = "round 1 (system 1)", "average of 3 (system 2)", "option A"
R1_CUTOFF = 0.75

def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def read_jsonl(path):
    with Path(path).open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

def macro_score(gold, predictions):
    """Official metric: per word, the mean of Yes-F1 and No-F1; then the mean over words."""
    report = positive_score(gold, predictions)
    languages = defaultdict(list)
    for row in report["per_word"]:
        tp, fp, fn = row["tp"], row["fp"], row["fn"]
        tn = row["n"] - tp - fp - fn
        parts = []
        if tp + fp + fn > 0:
            parts.append(2 * tp / (2 * tp + fp + fn))
        if tn + fp + fn > 0:
            parts.append(2 * tn / (2 * tn + fn + fp))
        row["positive_f1"], row["f1"] = row["f1"], sum(parts) / len(parts)
        languages[row["language"]].append(row["f1"])
    report["per_language"] = {k: sum(v) / len(v) for k, v in languages.items()}
    report["average"] = sum(report["per_language"].values()) / len(report["per_language"])
    return report

def pooled_macro(y, s, t):
    y, p = np.asarray(y), (np.asarray(s) >= t).astype(int)
    f = lambda c: 2 * np.sum((y == c) & (p == c)) / max(np.sum(y == c) + np.sum(p == c), 1)
    return float((f(1) + f(0)) / 2)

# ---------------------------------------------------------------------------
section("1. LOAD THE SCORES")
manifest = json.loads((ROOT / "data/manifest.json").read_text())
GROUPS, RULE = manifest["groups"], manifest["rule"]
R1_SV, ENS_SV = manifest["system1_swedish"], manifest["system2_swedish"]
sv = load_dataset(PROJECT / "work/data/dev", with_gold=True)
gold = sv.require_gold()
keys = sorted(gold)
word_group = {w: g for g, ws in GROUPS.items() for w in ws}
done = {}
for g in FOLDS:
    p = ROOT / f"fold_{g}/COMPLETE.json"
    assert p.exists(), f"Fold {g} is not finished: run cell 2 until its SUMMARY says COMPLETE for both folds."
    done[g] = json.loads(p.read_text())

def by_key(saved, tag):
    assert saved["model_tag"] == tag, f"Unexpected scores: {saved['model_tag']}"
    return {u.key: float(saved["scores"]["|".join(u.key)]["margin"]) for u in sv.usages}
r1 = by_key(json.loads((R1_RUN / "SV_scores.json").read_text()), "chosen_532_SV")
seed_scores = [r1] + [by_key(json.loads((ENS_ROOT / f"seed_{s}/eval_scores/SV_dev.json").read_text()), f"seed_{s}_SV_dev")
                      for s in (2028, 2029)]
ens = {k: float(np.mean([s[k] for s in seed_scores])) for k in keys}
oof, fold_of = {}, {}
for g, other in FOLDS.items():
    saved = json.loads((ROOT / f"fold_{g}/eval_scores/SV_heldout.json").read_text())
    assert saved["model_tag"] == f"fold_{g}_SV_heldout", f"Unexpected scores: {saved['model_tag']}"
    for u in sv.usages:
        if word_group[u.word] == other:
            assert u.key not in oof
            oof[u.key] = float(saved["scores"]["|".join(u.key)]["margin"])
            fold_of[u.key] = g
assert set(oof) == set(gold), "Some Swedish usages have no out-of-fold score: STOP and paste this output."
print(f"Option A Swedish scores: {len(oof):,} usages, each scored by the fold model that never saw that word")
print(f"Comparison scores loaded: round 1 (system 1) and the average of 3 (system 2)")

# ---------------------------------------------------------------------------
section("2. HEALTH CHECKS (a model that fails any of these is dropped, whatever its score)")
frozen_cfg = json.loads(FROZEN.read_text())
health, shas = {}, {}
for g in FOLDS:
    run = ROOT / f"fold_{g}"
    cfg = json.loads((run / "config.json").read_text())
    log = json.loads((run / "training_log.json").read_text())
    losses = [x["loss"] for x in log]
    first, last = float(np.mean(losses[:50])), float(np.mean(losses[-50:]))
    shas[g] = digest(Path(done[g]["adapter"]) / "adapter_model.safetensors")
    held = [k for k in keys if fold_of[k] == g]
    s_held = np.array([oof[k] for k in held])
    corr = float(np.corrcoef(s_held, [r1[k] for k in held])[0, 1])
    shift = float(np.mean(s_held) - np.mean([r1[k] for k in held]))
    health[g] = {
        f"all {cfg['steps']} updates in the log": len(log) == cfg["steps"],
        f"loss went down ({first:.3f} -> {last:.3f})": last < first,
        "adapter file intact (fingerprint)": shas[g] == done[g]["adapter_sha256"],
        "adapter differs from round 1": shas[g] != frozen_cfg["adapter_sha256"],
        f"Swedish answers differ from round 1 (correlation {corr:.3f} < 0.999)": corr < 0.999,
        "Swedish scores finite and in range": bool(np.isfinite(s_held).all() and np.abs(s_held).max() < 100),
    }
    print(f"Fold {g}:")
    for name, ok in health[g].items():
        print(f"   {'PASS' if ok else 'FAIL'}  {name}")
    print(f"   info  average Swedish score vs round 1 on the same usages: {shift:+.2f} (positive = leans more to Yes)")
two_different = shas["A"] != shas["B"]
print(f"{'PASS' if two_different else 'FAIL'}  the two fold adapters are different files")

# ---------------------------------------------------------------------------
if not SCORER_DIR.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/ChangeIsKey/semeval2027-task3.git", str(SCORER_DIR)], check=True)
subprocess.run(["git", "-C", str(SCORER_DIR), "checkout", "-q", SCORER_COMMIT], check=True)
assert "average='macro'" in (SCORER_DIR / "evaluation.py").read_text()
sys.path.insert(0, str(SCORER_DIR))
spec = importlib.util.spec_from_file_location("official_eval_latest", SCORER_DIR / "evaluation.py")
official = importlib.util.module_from_spec(spec)
spec.loader.exec_module(official)
def official_check(predictions):
    return official.compute_f1_score({f"{k[1]};{k[2]}": int(v) for k, v in gold.items()},
                                     {f"{k[1]};{k[2]}": int(predictions[k]) for k in gold}, "check")

section("3. SWEDISH DEV, OFFICIAL METRIC (nested leave-one-word-out, same procedure as before)")
def nested(scores, name):
    cv.score = macro_score
    try:
        return nested_word_cv(sv, [Candidate(name, frozen(scores), GRID)])
    finally:
        cv.score = positive_score
res = {R1N: nested(r1, "round1"), ENSN: nested(ens, "ensemble"), OPTN: nested(oof, "optionA")}
for name in res:
    assert abs(official_check(res[name]["predictions"]) - res[name]["report"]["average"]) < 1e-12, f"{name}: official scorer disagrees: STOP."
assert abs(res[R1N]["report"]["average"] - R1_SV) < 1e-9, "System 1 no longer reproduces: STOP and paste this output."
assert abs(res[ENSN]["report"]["average"] - ENS_SV) < 1e-9, "System 2 no longer reproduces: STOP and paste this output."
print(f"System 1 reproduces {R1_SV:.4f} and system 2 reproduces {ENS_SV:.4f}: PASS | official scorer agrees on all three: PASS\n")
for name in res:
    print(f"   {name:<26} macro-F1 {res[name]['report']['average']:.4f}")
per = {name: {row["word"]: row["f1"] for row in res[name]["report"]["per_word"]} for name in res}
words = sorted(per[OPTN])
better = sum(per[OPTN][w] > per[ENSN][w] + 1e-9 for w in words)
worse = sum(per[OPTN][w] < per[ENSN][w] - 1e-9 for w in words)
print(f"\n{'word':<11}{'scored by':>10}{'system 1':>10}{'system 2':>10}{'option A':>10}{'A minus sys 2':>15}")
for w in words:
    print(f"{w:<11}{'fold ' + FOLDS[word_group[w]]:>10}{per[R1N][w]:>10.3f}{per[ENSN][w]:>10.3f}{per[OPTN][w]:>10.3f}"
          f"{per[OPTN][w] - per[ENSN][w]:>+15.3f}")
paired = word_bootstrap(res[OPTN]["report"], other=res[ENSN]["report"])
print(f"\nOption A minus system 2: {paired['estimate']:+.4f} (95% range {paired['lower']:+.4f} to {paired['upper']:+.4f}) | "
      f"better on {better}, worse on {worse}, same on {10 - better - worse} of 10 words")

def fitted(scores):
    s = np.array([scores[k] for k in keys])
    t = max(GRID, key=lambda x: (macro_score(gold, dict(zip(keys, (s >= x).astype(int).tolist())))["average"], -x))
    return float(t), macro_score(gold, dict(zip(keys, (s >= t).astype(int).tolist())))["average"]
cut_A, insample_A = fitted(oof)
print(f"Option A cutoff fitted on all 10 words: {cut_A:+.2f} (in-sample {insample_A:.4f}); used for the other languages below")
both_answers = {}
for g in FOLDS:
    p = [oof[k] >= cut_A for k in keys if fold_of[k] == g]
    both_answers[g] = 0 < sum(p) < len(p)
    print(f"{'PASS' if both_answers[g] else 'FAIL'}  fold {g} predicts both Yes and No ({sum(p):,} Yes of {len(p):,})")

# ---------------------------------------------------------------------------
section("4. THE OTHER FIVE LANGUAGES: is any language worse? (each fold model vs round 1; AUC and macro-F1)")
ext, all_ok = {}, True
print(f"{'data':<26}{'AUC r1':>8}{'mF1 r1':>8}   {'AUC A':>7}{'mF1 A':>7}{'rule':>6}   {'AUC B':>7}{'mF1 B':>7}{'rule':>6}")
for name in EXTERNAL:
    rows = read_jsonl((ENRU_EVAL if name[:2] in ("EN", "RU") else XL_EVAL) / f"{name}.jsonl")
    ids, y = [str(row["id"]) for row in rows], [int(row["label"]) for row in rows]
    if name[:2] in ("EN", "RU"):
        base = json.loads((R1_RUN / f"{name}_scores.json").read_text())["scores"]
        base = [base[i]["margin"] for i in ids]
    else:
        base = json.loads((XL_SCORES / f"{name[:2]}_trained_scores.json").read_text())
        base = [base[i] for i in ids]
    a1, f1 = float(roc_auc_score(y, base)), pooled_macro(y, base, R1_CUTOFF)
    ext[name] = dict(auc_round1=a1, macro_f1_round1=f1)
    line = f"{name:<26}{a1:>8.4f}{f1:>8.4f}   "
    for g in FOLDS:
        saved = json.loads((ROOT / f"fold_{g}/eval_scores/{name}.json").read_text())
        assert saved["model_tag"] == f"fold_{g}_{name}", f"Unexpected scores: {saved['model_tag']}"
        s = np.array([saved["scores"][i]["margin"] for i in ids])
        assert np.isfinite(s).all(), f"fold {g} {name}: non-finite scores: STOP."
        a, f = float(roc_auc_score(y, s)), pooled_macro(y, s, cut_A)
        ok = bool(a - a1 >= -0.02 and f - f1 >= -0.02)
        all_ok &= ok
        ext[name][f"fold_{g}"] = dict(auc=a, macro_f1=f, rule_pass=ok)
        line += f"{a:>7.4f}{f:>7.4f}{'PASS' if ok else 'FAIL':>6}   "
    print(line)
print("Round 1 uses its cutoff 0.75; option A uses its own Swedish cutoff. External sets are not competition data.")

# ---------------------------------------------------------------------------
section("5. THE RULE (written down in cell 1, before training)")
opt_sv = res[OPTN]["report"]["average"]
health_ok = bool(all(all(h.values()) for h in health.values()) and two_different and all(both_answers.values()))
r1a, r1b = opt_sv >= ENS_SV + 0.02, better >= 6
print(f"Health   both fold models pass every health check            : {'PASS' if health_ok else 'FAIL'}")
print(f"Rule 1a  Swedish macro-F1 >= {ENS_SV + 0.02:.4f} (system 2 + 0.02)        : {opt_sv:.4f} -> {'PASS' if r1a else 'FAIL'}")
print(f"Rule 1b  better than system 2 on >= 6 of 10 Swedish words    : {better}/10   -> {'PASS' if r1b else 'FAIL'}")
print(f"Rule 2   no other language loses > 0.02 AUC or macro-F1      : {'PASS' if all_ok else 'FAIL'}")
adopt = bool(health_ok and r1a and r1b and all_ok)
save_json(ROOT / "option_a_decision.json", dict(
    metric="per-word macro-F1, official scorer commit 0e69238, nested leave-one-word-out on out-of-fold scores",
    swedish={name: res[name]["report"]["average"] for name in res}, per_word=per,
    paired_optionA_minus_system2=paired, words_better=int(better), words_worse=int(worse),
    cutoff=dict(cutoff=cut_A, in_sample=insample_A), health={g: {k: bool(v) for k, v in h.items()} for g, h in health.items()},
    both_answers={g: bool(v) for g, v in both_answers.items()}, adapters=shas, external=ext,
    rules=dict(health=health_ok, r1a=bool(r1a), r1b=bool(r1b), r2=bool(all_ok)), rule_text=RULE, adopt=adopt,
    note="Records the decision only. Nothing is frozen or replaced by this cell."))

section("DECISION")
if adopt:
    print(f"OPTION A PASSES: Swedish {ENS_SV:.4f} -> {opt_sv:.4f}, no language worse, all health checks pass.")
    print("Next: train one final model on round 1 data + all 10 Swedish words (about 1 hour on the G4). Keep the G4 connected.")
else:
    print(f"DROP OPTION A. It scored {opt_sv:.4f} on Swedish (system 1 {R1_SV:.4f}, system 2 {ENS_SV:.4f}).")
    print("System 1 and system 2 stay exactly as they are. Disconnect the G4 now to save units.")
print("Nothing was frozen or changed. Saved:", ROOT / "option_a_decision.json")

In [ ]:
"""CUTOFF CHECK (CPU only, about 3 minutes). Nothing is trained and no model or frozen file is changed.
Question: does a per-word Yes/No line (built from each word's own scores, no answers used) beat the single cutoff 0.75 for system 2?"""
import sys, json, subprocess, importlib.util
from collections import defaultdict
from pathlib import Path
import numpy as np
from google.colab import drive

drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/semeval2027/semeval2027_colab")
for folder in (PROJECT / "src", PROJECT / "notebooks"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bcubed"], check=True)
from project_utils import save_json, frozen
from semeval_task3.data import load_dataset
import semeval_task3.cv as cv
from semeval_task3.cv import Candidate, nested_word_cv
from semeval_task3.evaluation import score as positive_score, word_bootstrap

def section(t):
    print("\n" + "=" * 78 + f"\n{t}\n" + "=" * 78, flush=True)

R1_RUN = PROJECT / "outputs/runs/step09_lora_full_pilot/20261105T232211Z_6b04a51b"
ENS_ROOT = PROJECT / "outputs/runs/step15_seed_ensemble"
OUT = PROJECT / "outputs/runs/step17_cutoff_check"
SCORER_DIR, SCORER_COMMIT = Path("/content/official-scorer-latest"), "0e69238"
GRID = tuple([-1000.0] + np.round(np.arange(-100, 100.001, 0.25), 2).tolist() + [1000.0])
RULE = dict(
    compared="system 2 scores (average of 3 models); only the Yes/No line changes",
    options="single cutoff (now) | centered per word (score minus the word's median score) | half-centered per word (score minus half the word's median)",
    judged_number="nested leave-one-word-out where the option AND the cutoff are chosen inside each fold (no peeking at the held-out word)",
    adopt_if="judged number >= system 2 + 0.02 AND better than system 2 on >= 6 of 10 words",
    otherwise="keep the single cutoff 0.75")

def macro_score(gold, predictions):
    """Official metric: per word, the mean of Yes-F1 and No-F1; then the mean over words."""
    report = positive_score(gold, predictions)
    languages = defaultdict(list)
    for row in report["per_word"]:
        tp, fp, fn = row["tp"], row["fp"], row["fn"]
        tn = row["n"] - tp - fp - fn
        parts = []
        if tp + fp + fn > 0:
            parts.append(2 * tp / (2 * tp + fp + fn))
        if tn + fp + fn > 0:
            parts.append(2 * tn / (2 * tn + fn + fp))
        row["positive_f1"], row["f1"] = row["f1"], sum(parts) / len(parts)
        languages[row["language"]].append(row["f1"])
    report["per_language"] = {k: sum(v) / len(v) for k, v in languages.items()}
    report["average"] = sum(report["per_language"].values()) / len(report["per_language"])
    return report

def nested(candidates):
    cv.score = macro_score
    try:
        return nested_word_cv(sv, [Candidate(name, frozen(scores), GRID) for name, scores in candidates])
    finally:
        cv.score = positive_score

# ---------------------------------------------------------------------------
section("0. THE RULE (fixed before looking at any result)")
for k, v in RULE.items():
    print(f"{k:<14}: {v}")

section("1. LOAD SYSTEM 2 SWEDISH SCORES")
sv = load_dataset(PROJECT / "work/data/dev", with_gold=True)
gold = sv.require_gold()
keys = sorted(gold)
ENS_SV = json.loads((ENS_ROOT / "ensemble_decision.json").read_text())["swedish"]["ensemble (average of 3)"]
def by_key(saved, tag):
    assert saved["model_tag"] == tag, f"Unexpected scores: {saved['model_tag']}"
    return {u.key: float(saved["scores"]["|".join(u.key)]["margin"]) for u in sv.usages}
seeds = [by_key(json.loads((R1_RUN / "SV_scores.json").read_text()), "chosen_532_SV")] + \
        [by_key(json.loads((ENS_ROOT / f"seed_{s}/eval_scores/SV_dev.json").read_text()), f"seed_{s}_SV_dev") for s in (2028, 2029)]
ens = {k: float(np.mean([s[k] for s in seeds])) for k in keys}
by_word = defaultdict(list)
for k in keys:
    by_word[k[1]].append(ens[k])
med = {w: float(np.median(v)) for w, v in by_word.items()}
print(f"{len(keys):,} Swedish usages, {len(by_word)} words, scores = average of 3 models (system 2)\n")
print(f"{'word':<11}{'usages':>7}{'median score':>14}{'true Yes share':>16}   (Yes share shown for information only; never used by the rule)")
for w in sorted(by_word):
    ks = [k for k in keys if k[1] == w]
    print(f"{w:<11}{len(ks):>7}{med[w]:>+14.2f}{np.mean([gold[k] for k in ks]):>16.1%}")

variants = {
    "single cutoff (now)": ens,
    "centered per word": {k: ens[k] - med[k[1]] for k in keys},
    "half-centered per word": {k: ens[k] - 0.5 * med[k[1]] for k in keys},
}

# ---------------------------------------------------------------------------
if not SCORER_DIR.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/ChangeIsKey/semeval2027-task3.git", str(SCORER_DIR)], check=True)
subprocess.run(["git", "-C", str(SCORER_DIR), "checkout", "-q", SCORER_COMMIT], check=True)
assert "average='macro'" in (SCORER_DIR / "evaluation.py").read_text()
sys.path.insert(0, str(SCORER_DIR))
spec = importlib.util.spec_from_file_location("official_eval_latest", SCORER_DIR / "evaluation.py")
official = importlib.util.module_from_spec(spec)
spec.loader.exec_module(official)
def official_check(predictions):
    return official.compute_f1_score({f"{k[1]};{k[2]}": int(v) for k, v in gold.items()},
                                     {f"{k[1]};{k[2]}": int(predictions[k]) for k in gold}, "check")

section("2. EACH OPTION ON ITS OWN (for information; nested, official metric)")
alone = {name: nested([(f"v{i}", scores)]) for i, (name, scores) in enumerate(variants.items())}
for name, r in alone.items():
    assert abs(official_check(r["predictions"]) - r["report"]["average"]) < 1e-12, f"{name}: official scorer disagrees: STOP."
    print(f"   {name:<24} macro-F1 {r['report']['average']:.4f}")
assert abs(alone["single cutoff (now)"]["report"]["average"] - ENS_SV) < 1e-9, "System 2 no longer reproduces 0.7664: STOP and paste this output."
print(f"\nSystem 2 reproduces {ENS_SV:.4f}: PASS | official scorer agrees on all three: PASS")
print("Note: picking the best line above would be peeking. The judged number is in section 3.")

# ---------------------------------------------------------------------------
section("3. THE JUDGED NUMBER: option and cutoff both chosen inside each fold")
short = {"single cutoff (now)": "single", "centered per word": "centered", "half-centered per word": "half"}
chosen = nested([(short[n], s) for n, s in variants.items()])
assert abs(official_check(chosen["predictions"]) - chosen["report"]["average"]) < 1e-12, "Official scorer disagrees: STOP."
sel_sv = chosen["report"]["average"]
print(f"Per-word procedure (honest, nested): macro-F1 {sel_sv:.4f}   vs system 2 now {ENS_SV:.4f}   ({sel_sv - ENS_SV:+.4f})\n")
print("What each fold picked (held-out word: option, cutoff):")
for s in chosen.get("selections", []):
    print("   ", {k: v for k, v in s.items() if k in ("test_group", "candidate", "name", "threshold")})
base = {row["word"]: row["f1"] for row in alone["single cutoff (now)"]["report"]["per_word"]}
new = {row["word"]: row["f1"] for row in chosen["report"]["per_word"]}
better = sum(new[w] > base[w] + 1e-9 for w in base)
worse = sum(new[w] < base[w] - 1e-9 for w in base)
print(f"\n{'word':<11}{'system 2 now':>14}{'per-word':>10}{'change':>9}")
for w in sorted(base):
    print(f"{w:<11}{base[w]:>14.3f}{new[w]:>10.3f}{new[w] - base[w]:>+9.3f}")
paired = word_bootstrap(chosen["report"], other=alone["single cutoff (now)"]["report"])
print(f"\nChange: {paired['estimate']:+.4f} (95% range {paired['lower']:+.4f} to {paired['upper']:+.4f}) | "
      f"better on {better}, worse on {worse}, same on {len(base) - better - worse} of {len(base)} words")

# ---------------------------------------------------------------------------
section("4. THE RULE")
r_a, r_b = sel_sv >= ENS_SV + 0.02, better >= 6
print(f"Rule a  judged macro-F1 >= {ENS_SV + 0.02:.4f} (system 2 + 0.02)   : {sel_sv:.4f} -> {'PASS' if r_a else 'FAIL'}")
print(f"Rule b  better than system 2 on >= 6 of 10 words      : {better}/10   -> {'PASS' if r_b else 'FAIL'}")
adopt = bool(r_a and r_b)
OUT.mkdir(parents=True, exist_ok=True)
save_json(OUT / "cutoff_check.json", dict(
    rule=RULE, system2_swedish=ENS_SV, alone={n: r["report"]["average"] for n, r in alone.items()},
    judged=sel_sv, paired=paired, words_better=int(better), words_worse=int(worse),
    per_word=dict(system2=base, per_word_rule=new), word_medians=med, adopt=adopt,
    note="Other-language sets are always 50% Yes and built differently from the competition data, so they cannot judge a per-word rule."))

section("DECISION")
if adopt:
    print(f"ADOPT THE PER-WORD LINE: Swedish {ENS_SV:.4f} -> {sel_sv:.4f}. It goes into the submission pipeline.")
else:
    print(f"KEEP THE SINGLE CUTOFF 0.75. The per-word procedure scored {sel_sv:.4f} vs {ENS_SV:.4f}.")
print("No model or frozen file was changed. Saved:", OUT / "cutoff_check.json")